In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

### 导入必要的map参数

In [36]:
# 物流的渠道对照关系清洗用
month = 202512
Channel_map = {
'工程': '工程',
'零售':'零售',
'电商不可售':'电商',
'电商':'电商',
'内部处理通用':'电商',
'借出渠道':'非零售工程电商',
'新品':'电商',
'战略电商':'电商',
'转出渠道':'非零售工程电商',
'每誉':'每誉',
'渠道':'非零售工程电商',
'海外':'海外',
'调出渠道':'非零售工程电商',
'非零售工程电商':'非零售工程电商',
'非零售工程电商':'非零售工程电商'
}
#用于合并计算
productgroupset_map = {
    '吸油烟机':['吸油烟机'],
    '灶具':['灶具'],
    '烤箱':['烤箱'],
    '蒸箱':['蒸箱'],
    '微波炉':['微波炉'],
    '蒸烤烹饪机':['蒸烤烹饪机'],
    '蒸烤微烹饪机':['蒸烤微烹饪机'],
    '蒸微':['蒸微'],
    '蒸烤微合计':['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶消烹饪机':['灶消烹饪机'],
    '灶蒸烹饪机':['灶蒸烹饪机'],
    '灶蒸烤烹饪机':['灶蒸烤烹饪机'],
    '灶烤烹饪机':['灶烤烹饪机'],
    '灶集成':['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '烹饪产品线合计':['灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜':['消毒柜'],
    '热水器':['热水器'],
    '两用炉':['两用炉'],
    '热水器两用炉合计':['热水器','两用炉'],
    '家用净水机':['家用净水机'],
    '商用净水机':['商用净水机'],
    '净热产品线合计':['热水器','两用炉','家用净水机','商用净水机'],
    '水槽洗碗机':['水槽洗碗机'],
    '嵌入式洗碗机':['嵌入式洗碗机'],
    '洗碗机产品线合计':['水槽洗碗机','嵌入式洗碗机'],
    '国内合计':['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','家用净水机','商用净水机','水槽洗碗机','嵌入式洗碗机'],
}
# 统计的产品组
productgroup_list = ['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','热水器两用炉合计','家用净水机','商用净水机','净热产品线合计','水槽洗碗机','嵌入式洗碗机','洗碗机产品线合计','国内合计']


### 数据源处理-合并发货和财务的数据

In [37]:
#将这些excel的表格数据进行上下拼接，只要字段：商品编码、商品名称、渠道、实际出库数量
folder_path = fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\物流发货"  # 请替换为实际的文件夹路径
excel_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        sheet_names = file[5:].replace(file[-5:],'')
        if (file.endswith('.xlsx') or file.endswith('.xls')) and not file.startswith('~$'):
            file_path = os.path.join(root, file)
            excel_files.append((file, sheet_names, file_path))
print(excel_files)

[('2025年10月明细.xlsx', '10月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年10月明细.xlsx'), ('2025年11月明细.xlsx', '11月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年11月明细.xlsx'), ('2025年1月明细.xlsx', '1月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年1月明细.xlsx'), ('2025年2月明细.xlsx', '2月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年2月明细.xlsx'), ('2025年3月明细.xlsx', '3月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年3月明细.xlsx'), ('2025年4月明细.xlsx', '4月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年4月明细.xlsx'), ('2025年5月明细.xlsx', '5月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年5月明细.xlsx'), ('2025年6月明细.xlsx', '6月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年6月明细.xlsx'), ('2025年7月明细.xlsx', '7月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年7月明细.xlsx'), ('2025年8月明细.xlsx', '8月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年8月明细.xlsx'), ('2025年9月明细.xlsx', '9月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年9月明细.xlsx')]


In [38]:
#如果有报错请提示报错信息
df = pd.DataFrame()
for file in excel_files:
    try:
        df_temp = pd.read_excel(file[2], sheet_name=file[1])
        print(f'成功读取{file[0]}的{file[1]}表')
    except:
        print(f'读取{file[0]}的{file[1]}表失败')
    if '实际总数量' in df_temp.columns:
        df_temp = df_temp.rename(columns={'实际总数量':'实际出库数量'})
    df_temp = df_temp[['商品编码', '渠道', '实际出库数量']]
    df = pd.concat([df, df_temp], axis=0).reset_index(drop=True)
df = df.dropna(how='all').reset_index(drop=True)  # 仅当一行所有值都是NaN时才删除
df['商品编码'] = df['商品编码'].astype(str)
df['渠道'].value_counts()

成功读取2025年10月明细.xlsx的10月明细表
成功读取2025年11月明细.xlsx的11月明细表
成功读取2025年1月明细.xlsx的1月明细表
成功读取2025年2月明细.xlsx的2月明细表
成功读取2025年3月明细.xlsx的3月明细表
成功读取2025年4月明细.xlsx的4月明细表
成功读取2025年5月明细.xlsx的5月明细表
成功读取2025年6月明细.xlsx的6月明细表
成功读取2025年7月明细.xlsx的7月明细表
成功读取2025年8月明细.xlsx的8月明细表
成功读取2025年9月明细.xlsx的9月明细表


渠道
零售        439020
工程         18943
电商          9050
电商不可售       6616
海外          1924
每誉           155
战略电商         129
新品            67
内部处理通用        15
借出渠道          14
商净             7
渠道             4
调出渠道           2
转出渠道           2
米博新零售          1
Name: count, dtype: int64

In [39]:
#将这些excel的表格数据进行上下拼接，只要字段：商品编码、商品名称、渠道、实际出库数量
folder_path = fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\财务发货"  # 请替换为实际的文件夹路径
excel_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        sheet_names = file[5:].replace(file[-5:],'')
        if (file.endswith('.xlsx') or file.endswith('.xls')) and not file.startswith('~$'):
            file_path = os.path.join(root, file)
            excel_files.append((file, sheet_names, file_path))
print(excel_files)

[('2024年12月明细.xlsx', '12月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\财务发货\\2024年12月明细.xlsx')]


In [40]:
caiwu_shouru = {'商品编码':[],'渠道':[],'实际出库数量':[]}
for file in excel_files:
    try:
        df_temp = pd.read_excel(file[2], sheet_name=file[1])
        print(f'成功读取{file[0]}的{file[1]}表')
    except:
        print(file[0], file[1])
    for index,row in df_temp.iterrows():
        if row['零售'] > 0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('零售')
            caiwu_shouru['实际出库数量'].append(row['零售'])
        if row['工程'] >0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('工程')
            caiwu_shouru['实际出库数量'].append(row['工程'])
        if row['电商'] >0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('电商')
            caiwu_shouru['实际出库数量'].append(row['电商'])
        if row['合计-发货'] - row['零售'] - row['工程'] - row['电商'] != 0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('非零售工程电商')
            caiwu_shouru['实际出库数量'].append(row['合计-发货'] - row['零售'] - row['工程'] - row['电商'])

df_caiwu = pd.DataFrame(caiwu_shouru)
df_caiwu['商品编码'] = df_caiwu['商品编码'].astype(str)
df_caiwu['渠道'].value_counts()


成功读取2024年12月明细.xlsx的12月明细表


渠道
零售         521
电商         434
工程         253
非零售工程电商      8
Name: count, dtype: int64

In [41]:
df0 = pd.concat([df, df_caiwu], axis=0).reset_index(drop=True)
# df0.to_excel(r'C:\Users\zhangbon\Desktop\总发货.xlsx', index=False)
df0['物料编码'] = df0['商品编码'].astype(str)
df0['渠道'] = df0['渠道'].map(Channel_map).fillna('非零售工程电商')
df0 = df0[(df0['渠道']!='无') & 
        (df0['物料编码'].str.len()>=10)
        ].reset_index(drop=True)
df0['渠道'].value_counts()
# df0

渠道
零售         439541
工程          19196
电商          16311
海外           1924
每誉            155
非零售工程电商        16
Name: count, dtype: int64

### 数据处理-添加核算价，产品线、产品组、标准型号、销售时间、销售渠道、物料组（大系列）等信息

In [42]:
df0  = df0[['物料编码','渠道','实际出库数量']]
df0


,物料编码,渠道,实际出库数量
0,1001001500116,零售,12
1,1009000600033,零售,1
2,1009000500035,零售,3
3,1001001500131,零售,6
4,1002003700049,零售,5
...,...,...,...
477138,1021000400003,非零售工程电商,1866
477139,1021000400006,非零售工程电商,1644
477140,1021000400007,非零售工程电商,180
477141,1021000400008,非零售工程电商,2


In [43]:
# 引入物料组（大系列）
df_bigtype = pd.read_excel(r'C:\Users\zhangbon\Desktop\物料组对照（大系列）.xlsx')
df_bigtype['物料组'] = df_bigtype['物料组'].astype(str)
bigtype_map = dict(zip(df_bigtype['物料组'], df_bigtype['物料组描述']))
# bigtype_map

In [44]:
# PLM产品数据导入
df_plm = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx")
df_plm[['物料号','标准型号','国内/海外']] = df_plm[['物料号','标准型号','国内/海外']].astype(str)
df_plm['物料编码'] = df_plm['物料号'].apply(lambda x: x[:13])
df_plm['渠道'] = df_plm['下属渠道']
df_plm = df_plm[['物料编码','产品型号','标准型号','国内/海外','产品组','产品线','产品状态','渠道','开始销售时间','对应渠道状态']]
df_plm


e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,物料编码,产品型号,标准型号,国内/海外,产品组,产品线,产品状态,渠道,开始销售时间,对应渠道状态
0,nan,02-CS34BW,02-CS34BW,国内,灶具,烹饪厨电产品线,开发,零售,NaN,未售
1,nan,02-CS34BW,02-CS34BW,国内,灶具,烹饪厨电产品线,开发,电商,NaN,未售
2,nan,02-CS34BW,02-CS34BW,国内,灶具,烹饪厨电产品线,开发,工程,NaN,未售
3,nan,1,nan,国内,水槽洗碗机,洗碗机产品线,作废,NaN,NaN,NaN
4,1004000200090,10T-JSG15-0606FR,JSG15-0606,国内,热水器,净热产品线,停止发货,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
9212,nan,苍穹V5R1,苍穹V5R1,国内,吸油烟机,油烟机产品线,开发,NaN,NaN,NaN
9213,nan,轩辕V1R1,轩辕V1R1,国内,吸油烟机,油烟机产品线,量产,NaN,NaN,NaN
9214,nan,轩辕V1R1C02,轩辕V1R1C02,国内,吸油烟机,油烟机产品线,样机,NaN,NaN,NaN
9215,nan,轩辕V1R1C03,轩辕V1R1C03,国内,吸油烟机,油烟机产品线,开发,NaN,NaN,NaN


In [45]:
# 导入财务的核算价
df_price = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\核算价.xlsx")
df_price['物料编码'] = df_price['产品编码'].astype(str).str[:13]
df_price['系统核算价'] = df_price['系统核算价'].astype(float)
# df_price
df_price = df_price[['物料编码','系统核算价']]
df_price 

,物料编码,系统核算价
0,1001000100033,2056.0
1,1001000100034,1253.0
2,1001000100035,704.0
3,1001000100036,1990.0
4,1001000100037,1750.0
...,...,...
6733,9102000300001,5000.0
6734,9102000300002,5000.0
6735,9102000400000,5000.0
6736,9102000400001,5000.0


In [46]:
df0 = pd.merge(df0,df_plm,on=['物料编码','渠道'],how='left')
df0 = pd.merge(df0,df_price,on=['物料编码'],how='left')
df0['物料组'] = df0['物料编码'].str[:8]
df0['物料组描述'] = df0['物料组'].map(bigtype_map)
df0


,物料编码,渠道,实际出库数量,产品型号,标准型号,国内/海外,产品组,产品线,产品状态,开始销售时间,对应渠道状态,系统核算价,物料组,物料组描述
0,1001001500116,零售,12,CXW-358-Z8T(不带罩),Z8T,国内,吸油烟机,油烟机产品线,退市预警,12/10/2022 12:00:00 PM,退市预警,3358.0,10010015,整机油烟机Z系列
1,1009000600033,零售,1,ZK50-02-F1,ZK50-02-F1,国内,蒸烤烹饪机,烹饪厨电产品线,量产,6/9/2025 12:00:00 PM,在售,3450.0,10090006,整机烹饪机小蒸烤箱>
2,1009000500035,零售,3,JZT-ZK46-X2,JZT-ZK46-X2,国内,灶蒸烤烹饪机,烹饪厨电产品线,量产,4/29/2025 12:00:00 PM,在售,5280.0,10090005,整机烹饪机灶蒸烤A平>
3,1001001500131,零售,6,CXW-358-02-Z6TA(不带罩),02-Z6TA,国内,吸油烟机,油烟机产品线,退市预警,4/2/2024 12:00:00 PM,退市预警,2988.0,10010015,整机油烟机Z系列
4,1002003700049,零售,5,JZT-01-H8B-12T,H8B,国内,灶具,烹饪厨电产品线,量产,1/29/2024 12:00:00 PM,在售,2550.0,10020037,整机灶具H系列
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
477138,1021000400003,非零售工程电商,1866,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4200.0,10210004,整机手持式清洁机专>
477139,1021000400006,非零售工程电商,1644,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3346.0,10210004,整机手持式清洁机专>
477140,1021000400007,非零售工程电商,180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3522.0,10210004,整机手持式清洁机专>
477141,1021000400008,非零售工程电商,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3522.0,10210004,整机手持式清洁机专>


### 数据源处理-筛选渠道、产品线、产品组信息

In [47]:
df1 = df0[(df0['国内/海外']=='国内') & 
                # (df0['产品状态'].isin(['开发','停止生产','量产','停止销售','小批量','样机','退市预警','创建',])) &
                (df0['渠道'].isin(['零售','工程','电商']))&
                (df0['产品组'].isin(productgroup_list))
                ].reset_index(drop=True)
df1
df1['核算价'] = df1['系统核算价']*df1['实际出库数量']
print(f'没有系统核算价的是这些数据\n{df1[df1['系统核算价'].isnull()]['物料编码'].drop_duplicates()}')
print(len(df1))
display(df1)


没有系统核算价的是这些数据
Series([], Name: 物料编码, dtype: object)
446108


,物料编码,渠道,实际出库数量,产品型号,标准型号,国内/海外,产品组,产品线,产品状态,开始销售时间,对应渠道状态,系统核算价,物料组,物料组描述,核算价
0,1001001500116,零售,12,CXW-358-Z8T(不带罩),Z8T,国内,吸油烟机,油烟机产品线,退市预警,12/10/2022 12:00:00 PM,退市预警,3358.0,10010015,整机油烟机Z系列,40296.0
1,1009000600033,零售,1,ZK50-02-F1,ZK50-02-F1,国内,蒸烤烹饪机,烹饪厨电产品线,量产,6/9/2025 12:00:00 PM,在售,3450.0,10090006,整机烹饪机小蒸烤箱>,3450.0
2,1009000500035,零售,3,JZT-ZK46-X2,JZT-ZK46-X2,国内,灶蒸烤烹饪机,烹饪厨电产品线,量产,4/29/2025 12:00:00 PM,在售,5280.0,10090005,整机烹饪机灶蒸烤A平>,15840.0
3,1001001500131,零售,6,CXW-358-02-Z6TA(不带罩),02-Z6TA,国内,吸油烟机,油烟机产品线,退市预警,4/2/2024 12:00:00 PM,退市预警,2988.0,10010015,整机油烟机Z系列,17928.0
4,1002003700049,零售,5,JZT-01-H8B-12T,H8B,国内,灶具,烹饪厨电产品线,量产,1/29/2024 12:00:00 PM,在售,2550.0,10020037,整机灶具H系列,12750.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
446103,1013000500000,电商,7,YCZ-JT1600-HR7,YCZ-JT1800-HR7,国内,家用净水机,净热产品线,停止销售,6/8/2022 12:00:00 PM,停止发货,3640.0,10130005,整机净水机厨下净热>,25480.0
446104,1013000500001,零售,972,YCZ-JT1800-HR7,YCZ-JT1800-HR7,国内,家用净水机,净热产品线,停止销售,6/8/2022 12:00:00 PM,停止销售,3720.0,10130005,整机净水机厨下净热>,3615840.0
446105,1013000500001,工程,86,YCZ-JT1800-HR7,YCZ-JT1800-HR7,国内,家用净水机,净热产品线,停止销售,6/6/2022 12:00:00 PM,停止销售,3720.0,10130005,整机净水机厨下净热>,319920.0
446106,1013000100033,零售,61,YCZ-JT1800-01-M2E,YCZ-JT1800-01-M2E,国内,家用净水机,净热产品线,停止销售,8/28/2023 12:00:00 PM,停止销售,2280.0,10130001,整机净水机厨下净系列,139080.0


### 数据分析-产品组维度每个渠道的标准型号数、型号数、核算价总和、单型号贡献

In [48]:
df_temp1 = df1.copy()
# 创建辅助列
df_temp1['零售渠道标准型号'] = df_temp1.apply(lambda x: x['标准型号'] if x['渠道'] == '零售' else None, axis=1)
df_temp1['工程渠道标准型号'] = df_temp1.apply(lambda x: x['标准型号'] if x['渠道'] == '工程' else None, axis=1)
df_temp1['电商渠道标准型号'] = df_temp1.apply(lambda x: x['标准型号'] if x['渠道'] == '电商' else None, axis=1)

df_temp1['零售渠道核算价'] = df_temp1.apply(lambda x: x['核算价'] if x['渠道'] == '零售' else None, axis=1)
df_temp1['工程渠道核算价'] = df_temp1.apply(lambda x: x['核算价'] if x['渠道'] == '工程' else None, axis=1)
df_temp1['电商渠道核算价'] = df_temp1.apply(lambda x: x['核算价'] if x['渠道'] == '电商' else None, axis=1)

# 然后进行聚合
df_result1 = df_temp1.groupby('产品组', as_index=False).agg(
    标准型号数=('标准型号', 'nunique'),
    型号数=('物料编码', 'nunique'),
    核算价金额总计=('核算价', 'sum'),
    零售渠道标准型号数=('零售渠道标准型号', 'nunique'),
    零售渠道核算价=('零售渠道核算价', 'sum'),
    工程渠道标准型号数=('工程渠道标准型号', 'nunique'),
    工程渠道核算价=('工程渠道核算价', 'sum'),
    电商渠道标准型号数=('电商渠道标准型号', 'nunique'),
    电商渠道核算价=('电商渠道核算价', 'sum')
)
df_result1

,产品组,标准型号数,型号数,核算价金额总计,零售渠道标准型号数,零售渠道核算价,工程渠道标准型号数,工程渠道核算价,电商渠道标准型号数,电商渠道核算价
0,两用炉,6,6,5571882.0,5,5.496622e+06,1,7088.0,3,6.817200e+04
1,吸油烟机,165,250,7213337414.0,93,4.183673e+09,69,822357658.0,80,2.207307e+09
2,家用净水机,28,40,218364670.0,25,2.010544e+08,13,1967900.0,20,1.534235e+07
3,嵌入式洗碗机,63,71,1393376555.0,31,6.557507e+08,32,423278245.0,37,3.143476e+08
4,微波炉,5,6,18757080.0,4,1.193597e+07,3,2672610.0,2,4.148500e+06
5,水槽洗碗机,59,78,775024490.0,43,5.642080e+08,25,69506045.0,41,1.413105e+08
6,消毒柜,29,53,392213480.0,21,1.779971e+08,17,108287810.0,21,1.059285e+08
7,灶具,107,378,3621211710.0,67,1.903246e+09,60,431497682.0,53,1.286468e+09
8,灶消烹饪机,3,5,35481640.0,3,3.484094e+07,1,7540.0,3,6.331600e+05
9,灶蒸烤烹饪机,19,64,621574550.0,18,5.894398e+08,10,29534330.0,9,2.600390e+06


In [49]:
df_result1['产品组'] = pd.Categorical(
    df_result1['产品组'], 
    categories=productgroup_list, 
    ordered=True
)
df_result1 = df_result1.sort_values('产品组').reset_index(drop=True)
df_result1

,产品组,标准型号数,型号数,核算价金额总计,零售渠道标准型号数,零售渠道核算价,工程渠道标准型号数,工程渠道核算价,电商渠道标准型号数,电商渠道核算价
0,吸油烟机,165,250,7213337414.0,93,4.183673e+09,69,822357658.0,80,2.207307e+09
1,灶具,107,378,3621211710.0,67,1.903246e+09,60,431497682.0,53,1.286468e+09
2,烤箱,12,14,71565720.0,8,5.656097e+07,7,4735340.0,7,1.026941e+07
3,蒸箱,11,13,82676350.0,7,6.705725e+07,4,2196950.0,7,1.342215e+07
4,微波炉,5,6,18757080.0,4,1.193597e+07,3,2672610.0,2,4.148500e+06
5,蒸烤烹饪机,45,48,853117135.0,25,6.329581e+08,20,67408949.0,36,1.527501e+08
6,蒸烤微烹饪机,8,8,85824050.0,4,1.967123e+07,5,1585950.0,5,6.456687e+07
7,灶消烹饪机,3,5,35481640.0,3,3.484094e+07,1,7540.0,3,6.331600e+05
8,灶蒸烹饪机,2,2,156720.0,1,9.954000e+04,0,0.0,2,5.718000e+04
9,灶蒸烤烹饪机,19,64,621574550.0,18,5.894398e+08,10,29534330.0,9,2.600390e+06


### 数据分析-只看吸油烟机，看各个系列的单型号情况

In [50]:
df_temp2 = df1.copy()
df_temp2 = df_temp2[df_temp2['产品线']=='油烟机产品线']

#### 从物料组（大系列）角度分析

In [51]:
# 创建辅助列
df_temp2['零售渠道标准型号'] = df_temp2.apply(lambda x: x['标准型号'] if x['渠道'] == '零售' else None, axis=1)
df_temp2['工程渠道标准型号'] = df_temp2.apply(lambda x: x['标准型号'] if x['渠道'] == '工程' else None, axis=1)
df_temp2['电商渠道标准型号'] = df_temp2.apply(lambda x: x['标准型号'] if x['渠道'] == '电商' else None, axis=1)

df_temp2['零售渠道核算价'] = df_temp2.apply(lambda x: x['核算价'] if x['渠道'] == '零售' else None, axis=1)
df_temp2['工程渠道核算价'] = df_temp2.apply(lambda x: x['核算价'] if x['渠道'] == '工程' else None, axis=1)
df_temp2['电商渠道核算价'] = df_temp2.apply(lambda x: x['核算价'] if x['渠道'] == '电商' else None, axis=1)

df_temp2['零售渠道发货量'] = df_temp2.apply(lambda x: x['实际出库数量'] if x['渠道'] == '零售' else None, axis=1)
df_temp2['工程渠道发货量'] = df_temp2.apply(lambda x: x['实际出库数量'] if x['渠道'] == '工程' else None, axis=1)
df_temp2['电商渠道发货量'] = df_temp2.apply(lambda x: x['实际出库数量'] if x['渠道'] == '电商' else None, axis=1)
# 然后进行聚合
df_result2 = df_temp2.groupby('物料组描述', as_index=False).agg(
    标准型号数=('标准型号', 'nunique'),
    型号数=('物料编码', 'nunique'),
    销售数量=('实际出库数量', 'sum'),
    核算价金额总计=('核算价', 'sum'),
    零售渠道标准型号数=('零售渠道标准型号', 'nunique'),
    零售渠道发货量=('零售渠道发货量', 'sum'),
    零售渠道核算价=('零售渠道核算价', 'sum'),
    工程渠道标准型号数=('工程渠道标准型号', 'nunique'),
    工程渠道发货量=('工程渠道发货量', 'sum'),
    工程渠道核算价=('工程渠道核算价', 'sum'),
    电商渠道标准型号数=('电商渠道标准型号', 'nunique'),
    电商渠道发货量=('电商渠道发货量', 'sum'),
    电商渠道核算价=('电商渠道核算价', 'sum')
)
df_result2['单型号贡献'] = df_result2['核算价金额总计'] / df_result2['标准型号数']
df_result2['零售渠道单型号贡献平均值'] = (df_result2['零售渠道核算价'] / df_result2['零售渠道标准型号数']).fillna(0)
df_result2['工程渠道单型号贡献平均值'] = (df_result2['工程渠道核算价'] / df_result2['工程渠道标准型号数']).fillna(0)
df_result2['电商渠道单型号贡献平均值'] = (df_result2['电商渠道核算价'] / df_result2['电商渠道标准型号数']).fillna(0)
df_result2 = df_result2.sort_values(by='核算价金额总计', ascending=False).reset_index(drop=True)
df_result2

,物料组描述,标准型号数,型号数,销售数量,核算价金额总计,零售渠道标准型号数,零售渠道发货量,零售渠道核算价,工程渠道标准型号数,工程渠道发货量,工程渠道核算价,电商渠道标准型号数,电商渠道发货量,电商渠道核算价,单型号贡献,零售渠道单型号贡献平均值,工程渠道单型号贡献平均值,电商渠道单型号贡献平均值
0,整机油烟机Z系列,22,29,867050,2450584670.0,15,662799.0,1.861106e+09,7,23984.0,64613292.0,13,180267.0,524865096.0,111390212.272727,1.240738e+08,9.230470e+06,4.037424e+07
1,整机油烟机EM系列,40,75,785248,1604008630.0,20,283292.0,5.155207e+08,18,143951.0,267756916.0,19,358005.0,820731036.0,40100215.75,2.577603e+07,1.487538e+07,4.319637e+07
2,整机油烟机X系列,13,13,248025,801809268.0,11,223235.0,7.133206e+08,7,22390.0,80486620.0,6,2400.0,8002028.0,61677636.0,6.484733e+07,1.149809e+07,1.333671e+06
3,整机油烟机JC系列,31,40,310518,689234228.0,17,101944.0,2.167074e+08,6,30258.0,54530404.0,20,178316.0,417996448.0,22233362.193548,1.274749e+07,9.088401e+06,2.089982e+07
4,整机油烟机Y系列,5,5,100931,417313208.0,5,97547.0,4.010074e+08,3,3384.0,16305772.0,0,0.0,0.0,83462641.6,8.020149e+07,5.435257e+06,0.000000e+00
5,整机油烟机7字型系列,4,11,132183,311090044.0,1,19796.0,4.594303e+07,0,0.0,0.0,4,112387.0,265147016.0,77772511.0,4.594303e+07,0.000000e+00,6.628675e+07
6,整机油烟机EH系列,8,21,179918,284131904.0,2,5851.0,9.290388e+06,7,149107.0,233533436.0,3,24960.0,41308080.0,35516488.0,4.645194e+06,3.336192e+07,1.376936e+07
7,整机油烟机X20系列,5,5,52493,239846228.0,5,48271.0,2.203375e+08,2,3539.0,16381980.0,1,683.0,3126774.0,47969245.6,4.406749e+07,8.190990e+06,3.126774e+06
8,整机油烟机JQ系列,17,27,95312,191735162.0,9,25415.0,5.358728e+07,10,27762.0,59570788.0,8,42135.0,78577096.0,11278538.941176,5.954142e+06,5.957079e+06,9.822137e+06
9,整机油烟机J系列,4,4,44021,134907032.0,4,44009.0,1.348578e+08,0,0.0,0.0,1,12.0,49200.0,33726758.0,3.371446e+07,0.000000e+00,4.920000e+04


#### 从2080的角度分析

In [ ]:
# 从2080的角度分析一下
df_result3 = df_temp2.groupby(['物料编码'],as_index=False).agg(
                                            产品型号=('产品型号', 'min'),
                                            发货量=('实际出库数量', 'sum'),
                                            核算价金额总计=('核算价', 'sum'),
                                            标准型号=('标准型号', 'min'),
                                            产品状态=('产品状态', 'min'),
                                            物料组=('物料组', 'min'),
                                            物料组描述=('物料组描述', 'min'),
        
)
df_result3 = df_result3.sort_values(by='核算价金额总计', ascending=False).reset_index(drop=True)
df_result3['累计核算价'] = df_result3['核算价金额总计'].cumsum()
df_result3['80%收入额'] = df_result3['核算价金额总计'].sum() * 0.8
df_result3['是否主力型号'] = df_result3['累计核算价'] <= df_result3['80%收入额']
df_result3

,物料编码,产品型号,发货量,核算价金额总计,标准型号,产品状态,物料组,物料组描述,累计核算价,80%收入额,是否主力型号
0,1001001500097,CXW-358-Z7T（不带罩）,229993,604421604.0,Z7T,退市预警,10010015,整机油烟机Z系列,604421604.0,5.770670e+09,True
1,1001002000029,CXW-358-04-X2A,98799,310031262.0,04-X2A,量产,10010020,整机油烟机X系列,914452866.0,5.770670e+09,True
2,1001000900395,CXW-358-F3-G,121542,302396496.0,F3-G,量产,10010009,整机油烟机EM系列,1216849362.0,5.770670e+09,True
3,1001001500106,CXW-358-Z5TA（不带罩）,94923,283629924.0,Z5TA,退市预警,10010015,整机油烟机Z系列,1500479286.0,5.770670e+09,True
4,1001001500144,CXW-358-V1S-G,94383,272578104.0,V1S-G,量产,10010015,整机油烟机Z系列,1773057390.0,5.770670e+09,True
...,...,...,...,...,...,...,...,...,...,...,...
245,1001002800003,CXW-358-01-X20Q.i,2,10456.0,01-X20Q.i,退市预警,10010028,整机油烟机X20系列,7213308198.0,5.770670e+09,False
246,1001000900227,CXW-258-EMD16A,4,9032.0,EMD16A,停止发货,10010009,整机油烟机EM系列,7213317230.0,5.770670e+09,False
247,1001000500251,CXW-228-JQ15T(不带罩),3,7884.0,JQ15T,停止销售,10010005,整机油烟机JQ系列,7213325114.0,5.770670e+09,False
248,1001000900226,CXW-258-EM18A,3,6624.0,EM18A,停止销售,10010009,整机油烟机EM系列,7213331738.0,5.770670e+09,False


In [72]:
revenue_median = df_result3['核算价金额总计'].median()
revenue_mean = df_result3['核算价金额总计'].mean()
revenue_median,revenue_mean

(np.float64(8884196.0), np.float64(28853349.656))

In [ ]:
# 四象限分析
revenue_median = df_result3['核算价金额总计'].median()
revenue_mean = df_result3['核算价金额总计'].mean()
def classify_revenue(row,standard):
    if row['核算价金额总计'] >= standard and row['产品状态'] in ['量产']:
        return '明星产品'
    if row['核算价金额总计'] < standard and row['产品状态'] in ['量产']:
        return '潜力产品'
    if row['核算价金额总计'] >= standard and not row['产品状态'] in ['量产']:
        return '风险产品'
    if row['核算价金额总计'] < standard and not row['产品状态'] in ['量产']:
        return '问题产品'
df_result3['产品分类(中位数)'] = df_result3.apply(lambda row: classify_revenue(row,revenue_median), axis=1)
df_result3['产品分类(均值)'] = df_result3.apply(lambda row: classify_revenue(row,revenue_mean), axis=1)
df_result3


,物料编码,产品型号,发货量,核算价金额总计,标准型号,产品状态,物料组,物料组描述,累计核算价,80%收入额,是否主力型号,产品分类,产品分类(中位数),产品分类(均值)
0,1001001500097,CXW-358-Z7T（不带罩）,229993,604421604.0,Z7T,退市预警,10010015,整机油烟机Z系列,604421604.0,5.770670e+09,True,风险产品,风险产品,风险产品
1,1001002000029,CXW-358-04-X2A,98799,310031262.0,04-X2A,量产,10010020,整机油烟机X系列,914452866.0,5.770670e+09,True,明星产品,明星产品,明星产品
2,1001000900395,CXW-358-F3-G,121542,302396496.0,F3-G,量产,10010009,整机油烟机EM系列,1216849362.0,5.770670e+09,True,明星产品,明星产品,明星产品
3,1001001500106,CXW-358-Z5TA（不带罩）,94923,283629924.0,Z5TA,退市预警,10010015,整机油烟机Z系列,1500479286.0,5.770670e+09,True,风险产品,风险产品,风险产品
4,1001001500144,CXW-358-V1S-G,94383,272578104.0,V1S-G,量产,10010015,整机油烟机Z系列,1773057390.0,5.770670e+09,True,明星产品,明星产品,明星产品
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,1001002800003,CXW-358-01-X20Q.i,2,10456.0,01-X20Q.i,退市预警,10010028,整机油烟机X20系列,7213308198.0,5.770670e+09,False,问题产品,问题产品,问题产品
246,1001000900227,CXW-258-EMD16A,4,9032.0,EMD16A,停止发货,10010009,整机油烟机EM系列,7213317230.0,5.770670e+09,False,问题产品,问题产品,问题产品
247,1001000500251,CXW-228-JQ15T(不带罩),3,7884.0,JQ15T,停止销售,10010005,整机油烟机JQ系列,7213325114.0,5.770670e+09,False,问题产品,问题产品,问题产品
248,1001000900226,CXW-258-EM18A,3,6624.0,EM18A,停止销售,10010009,整机油烟机EM系列,7213331738.0,5.770670e+09,False,问题产品,问题产品,问题产品


In [74]:
with pd.ExcelWriter(fr'C:\Users\zhangbon\Desktop\探索分析.xlsx') as writer:
    df_result3.to_excel(writer, sheet_name='探索', index=False)

In [75]:
# 看一下非主力型号数据
df_result3.groupby('物料组描述',as_index=False).agg(
                                    非主力型号数 = ('是否主力型号', lambda x: (x == False).sum()),
                                    主力型号数 = ('是否主力型号', lambda x: (x == True).sum()),
                                    型号数 = ('是否主力型号', 'count'),
)

,物料组描述,非主力型号数,主力型号数,型号数
0,整机油烟机7字型系列,8,3,11
1,整机油烟机EA系列,1,0,1
2,整机油烟机EG系列,5,0,5
3,整机油烟机EH系列,18,3,21
4,整机油烟机EM系列,62,13,75
5,整机油烟机EN系列,1,0,1
6,整机油烟机JC系列,31,9,40
7,整机油烟机JQ系列,26,1,27
8,整机油烟机JX系列,1,0,1
9,整机油烟机J系列,2,2,4


In [56]:
# 看一下发货量小于200的整机
df_result3[df_result3['发货量'] < 200]

,物料编码,产品型号,发货量,核算价金额总计,标准型号,产品状态,物料组,物料组描述,累计核算价,80%收入额,是否主力型号
198,1001002800001,CXW-358-01-X20Pro,67,723600.0,01-X20Pro,退市预警,10010028,整机油烟机X20系列,7200199828.0,5.770670e+09,False
212,1001000200087,CXW-258-EA06,23,446200.0,EA06,量产,10010002,整机油烟机EA系列,7208459594.0,5.770670e+09,False
213,1001000500073,CXW-200-JQ01TS（不带罩）,148,422984.0,JQ01TS,停止销售,10010005,整机油烟机JQ系列,7208882578.0,5.770670e+09,False
214,1001000500361,CXW-258-JQ01TY(不带罩),112,420896.0,JQ01TY,停止销售,10010005,整机油烟机JQ系列,7209303474.0,5.770670e+09,False
217,1001002100058,CXW-258-01-F1A（不带罩）,133,353514.0,01-F1,停止销售,10010021,整机油烟机JC系列,7210458576.0,5.770670e+09,False
218,1001000900349,CXW-358-EMG7A,168,317184.0,EMG5A,量产,10010009,整机油烟机EM系列,7210775760.0,5.770670e+09,False
219,1001000500355,CXW-258-JQC2A（不带罩）,150,274500.0,JQC2A,退市预警,10010005,整机油烟机JQ系列,7211050260.0,5.770670e+09,False
220,1001001200002,CXW-200-EN05E(电商升级版),109,240672.0,EN05E,停止销售,10010012,整机油烟机EN系列,7211290932.0,5.770670e+09,False
221,1001000800358,CXW-258-EH39H,141,238008.0,EH36H,量产,10010008,整机油烟机EH系列,7211528940.0,5.770670e+09,False
222,1001001500095,CXW-258-ZD70（不带罩）,127,205486.0,ZD70,停止销售,10010015,整机油烟机Z系列,7211734426.0,5.770670e+09,False


In [57]:
# 看一下单型号贡献的平均值（标准型号维度）
avg_single_price = df_temp2['核算价'].sum() / df_temp2['标准型号'].nunique()
sum_income = df_temp2['核算价'].sum()
avg_single_price,sum_income

(43717196.448484845, 7213337414.0)

In [58]:
display(df_temp2)
df_temp2.columns

,物料编码,渠道,实际出库数量,产品型号,标准型号,国内/海外,产品组,产品线,产品状态,开始销售时间,...,核算价,零售渠道标准型号,工程渠道标准型号,电商渠道标准型号,零售渠道核算价,工程渠道核算价,电商渠道核算价,零售渠道发货量,工程渠道发货量,电商渠道发货量
0,1001001500116,零售,12,CXW-358-Z8T(不带罩),Z8T,国内,吸油烟机,油烟机产品线,退市预警,12/10/2022 12:00:00 PM,...,40296.0,Z8T,None,None,40296.0,NaN,NaN,12.0,NaN,NaN
3,1001001500131,零售,6,CXW-358-02-Z6TA(不带罩),02-Z6TA,国内,吸油烟机,油烟机产品线,退市预警,4/2/2024 12:00:00 PM,...,17928.0,02-Z6TA,None,None,17928.0,NaN,NaN,6.0,NaN,NaN
7,1001002000028,零售,5,CXW-358-04-X5A,04-X5A,国内,吸油烟机,油烟机产品线,量产,9/22/2023 12:00:00 PM,...,13190.0,04-X5A,None,None,13190.0,NaN,NaN,5.0,NaN,NaN
9,1001002000018,零售,2,CXW-358-03-X1A,03-X1A,国内,吸油烟机,油烟机产品线,量产,12/22/2022 12:00:00 PM,...,7336.0,03-X1A,None,None,7336.0,NaN,NaN,2.0,NaN,NaN
13,1001001500117,零售,6,CXW-358-Z5TS（不带罩）,Z5TS,国内,吸油烟机,油烟机产品线,量产,10/17/2022 12:00:00 PM,...,17808.0,Z5TS,None,None,17808.0,NaN,NaN,6.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
445294,1001001500129,工程,3,CXW-358-Z3.i(不带罩),Z3.i,国内,吸油烟机,油烟机产品线,量产,4/1/2024 12:00:00 PM,...,11544.0,None,Z3.i,None,NaN,11544.0,NaN,NaN,3.0,NaN
445295,1001001500129,电商,1,CXW-358-Z3.i(不带罩),Z3.i,国内,吸油烟机,油烟机产品线,量产,3/2/2024 12:00:00 PM,...,3848.0,None,None,Z3.i,NaN,NaN,3848.0,NaN,NaN,1.0
445296,1001000900394,电商,5600,CXW-358-F3,F3,国内,吸油烟机,油烟机产品线,量产,4/18/2024 12:00:00 PM,...,13932800.0,None,None,F3,NaN,NaN,13932800.0,NaN,NaN,5600.0
445297,1001000900395,电商,19125,CXW-358-F3-G,F3-G,国内,吸油烟机,油烟机产品线,量产,4/18/2024 12:00:00 PM,...,47583000.0,None,None,F3-G,NaN,NaN,47583000.0,NaN,NaN,19125.0


Index(['物料编码', '渠道', '实际出库数量', '产品型号', '标准型号', '国内/海外', '产品组', '产品线', '产品状态',
       '开始销售时间', '对应渠道状态', '系统核算价', '物料组', '物料组描述', '核算价', '零售渠道标准型号',
       '工程渠道标准型号', '电商渠道标准型号', '零售渠道核算价', '工程渠道核算价', '电商渠道核算价', '零售渠道发货量',
       '工程渠道发货量', '电商渠道发货量'],
      dtype='object')